# 02 Quality checks

## Cosa fa / Cosa NON fa

- esegue controlli riutilizzabili su duplicati, missingness e range
- prova a rilevare colonne numeriche in modo automatico
- se mancano chiavi o colonne adatte, stampa istruzioni invece di fallire

In [ ]:
from pathlib import Path
import duckdb

ROOT = Path('.').resolve()
TABLE_NAME = 'project_summary'
KEY_COLUMNS = []
NUMERIC_COLUMNS = []

def find_mart(table_name):
    for base in [(ROOT / '..' / 'data' / 'mart').resolve(), (ROOT / '..' / '_runs').resolve()]:
        if not base.exists():
            continue
        matches = sorted(base.glob(f'**/*{table_name}*.parquet'))
        if matches:
            return matches[0]
    return None

mart_path = find_mart(TABLE_NAME)
mart_path

In [ ]:
con = duckdb.connect()

def read_schema(path):
    schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()
    schema.columns = [str(col).lower() for col in schema.columns]
    return schema

def detect_numeric_columns(schema):
    name_col = 'column_name' if 'column_name' in schema.columns else schema.columns[0]
    type_col = 'column_type' if 'column_type' in schema.columns else schema.columns[1]
    numeric = []
    for _, row in schema.iterrows():
        dtype = str(row[type_col]).upper()
        if any(token in dtype for token in ['INT', 'DECIMAL', 'DOUBLE', 'FLOAT', 'REAL', 'BIGINT']):
            numeric.append(str(row[name_col]))
    return numeric

def duplicate_key_report(path, key_columns):
    keys = ', '.join(key_columns)
    sql = f"""
        SELECT {keys}, COUNT(*) AS dup_count
        FROM read_parquet('{path}')
        GROUP BY {keys}
        HAVING COUNT(*) > 1
        ORDER BY dup_count DESC
        LIMIT 20
    """
    return con.execute(sql).df()

def missingness_report(path):
    columns = [row[0] for row in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").fetchall()]
    expr = ', '.join([f"AVG(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS {col}_null_rate" for col in columns])
    return con.execute(f"SELECT {expr} FROM read_parquet('{path}')").df().T.reset_index()

def range_report(path, numeric_columns):
    expr = ', '.join([f"MIN({col}) AS {col}_min, MAX({col}) AS {col}_max" for col in numeric_columns])
    return con.execute(f"SELECT {expr} FROM read_parquet('{path}')").df().T.reset_index()

if mart_path:
    schema_df = read_schema(str(mart_path))
    detected_numeric = detect_numeric_columns(schema_df)
    numeric_for_range = [col for col in NUMERIC_COLUMNS if col in detected_numeric]
    if not numeric_for_range:
        numeric_for_range = detected_numeric[:3]
    display(schema_df)
    print({'KEY_COLUMNS': KEY_COLUMNS, 'NUMERIC_COLUMNS': numeric_for_range})

In [ ]:
if mart_path:
    if KEY_COLUMNS:
        dup_df = duplicate_key_report(str(mart_path), KEY_COLUMNS)
        display(dup_df)
    else:
        print('Duplicate-key check skipped: imposta KEY_COLUMNS.')

    missing_df = missingness_report(str(mart_path))
    display(missing_df)

    if numeric_for_range:
        range_df = range_report(str(mart_path), numeric_for_range)
        display(range_df)
    else:
        print('Range check skipped: no numeric columns detected.')